# Gradient Sports API — Exemplos de Uso

Importa o cliente de `gradient_client.py`.  
A autenticação é lida automaticamente do arquivo `.env` (`BEARER_TOKEN`).

In [1]:
from gradient_client import GradientSportsClient

## Usage examples

In [2]:
client = GradientSportsClient()

# health check
client.get_status()

{'data': {'status': 'ok'}}

In [3]:
# competitions & seasons the account can access
df_competitions = client.get_competitions(as_dataframe=True)
df_competitions

,datasets,season,competition.id,competition.name
0,physical_metrics,2020-2021,1,Premier League
1,events,2020-2021,1,Premier League
2,sprints,2020-2021,1,Premier League
3,high_speed_runs,2020-2021,1,Premier League
4,physical_metrics,2021-2022,1,Premier League
5,events,2021-2022,1,Premier League
6,sprints,2021-2022,1,Premier League
7,high_speed_runs,2021-2022,1,Premier League
8,physical_metrics,2022-2023,1,Premier League
9,events,2022-2023,1,Premier League


In [4]:
# teams the account can access
df_teams = client.get_teams(as_dataframe=True)
df_teams.tail()

,team.id,team.name,competition.id,competition.name,dataset
225,15.0,Sheffield United,1,Premier League,high_speed_runs
226,425.0,América Mineiro,42,Brasileiro Série A,physical_metrics
227,425.0,América Mineiro,42,Brasileiro Série A,events
228,425.0,América Mineiro,42,Brasileiro Série A,sprints
229,425.0,América Mineiro,42,Brasileiro Série A,high_speed_runs


In [5]:
# all games — or filter by season / competition / team
# as_dataframe=True → one row per game, nested dicts dot-expanded
games = client.get_games(season="2021-2022", competition_id=1, as_dataframe=True)
games.head()

,id,date,season,teamExtraTimeStartSide,teamStartSide,venueType,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,stadium.name,stadium.length,stadium.width
0,1449,2022-05-22,2021-2022,Right,Right,TEAM_HOME,6,Chelsea,1,Premier League,18,Watford,Stamford Bridge,103.0,67.5
1,1421,2022-05-01,2021-2022,Left,Left,OPPONENT_HOME,9,Leicester City,1,Premier League,17,Tottenham Hotspur,Tottenham Hotspur Stadium,105.0,68.0
2,1185,2021-11-20,2021-2022,Right,Left,TEAM_HOME,3,Aston Villa,1,Premier League,4,Brighton & Hove Albion,Villa Park,105.0,68.0
3,1127,2021-09-25,2021-2022,Right,Right,TEAM_HOME,6,Chelsea,1,Premier League,11,Manchester City,Stamford Bridge,103.0,67.5
4,1263,2021-12-26,2021-2022,Left,Right,OPPONENT_HOME,16,Southampton,1,Premier League,19,West Ham,London Stadium,105.0,68.0


In [6]:
# pick a game id from the list above
GAME_ID = games["id"].iloc[0]

# structured events — one row per possession event
df_game_events = client.get_game_events(GAME_ID, as_dataframe=True)
df_game_events = df_game_events.sort_values(by="poss.startGameClock")
df_game_events.head()

,id,competitionId,gameId,season,period,periodDescription,startGameClock,startFormattedGameClock,homeTeam,gameEventType,...,poss.endGameClock,poss.period,poss.nonEvent,poss.ballHeightType,poss.highPointType,poss.bodyType,poss.player.id,poss.player.name,poss.team.id,poss.team.name
0,3858104,None,None,None,1,First half,0,00:00,False,FIRSTKICKOFF,...,190.997,1,False,G,G,R,422.0,Tom Cleverley,18.0,Watford
1,3858111,None,None,None,1,First half,1,00:01,False,OTB,...,194.065,1,False,G,A,L,7143.0,Samir Caetano de Souza Santos,18.0,Watford
3,3858115,None,None,None,1,First half,6,00:06,False,OTB,...,197.166,1,False,A,A,HE,310.0,Joshua King,18.0,Watford
2,3858115,None,None,None,1,First half,6,00:06,False,OTB,...,197.166,1,False,NaN,NaN,NaN,310.0,Joshua King,18.0,Watford
4,3879275,None,None,None,1,First half,6,00:06,True,OTB,...,197.199,1,False,A,A,HE,1554.0,Robert do Nascimento,6.0,Chelsea


In [7]:
df_game_events.columns

Index(['id', 'competitionId', 'gameId', 'season', 'period',
       'periodDescription', 'startGameClock', 'startFormattedGameClock',
       'homeTeam', 'gameEventType', 'gameEventTypeDescription', 'setpieceType',
       'setpieceTypeDescription', 'touches', 'touchesInBox', 'team.id',
       'team.name', 'player.id', 'player.name', 'poss.id', 'poss.type',
       'poss.typeDescription', 'poss.startGameClock', 'poss.endGameClock',
       'poss.period', 'poss.nonEvent', 'poss.ballHeightType',
       'poss.highPointType', 'poss.bodyType', 'poss.player.id',
       'poss.player.name', 'poss.team.id', 'poss.team.name'],
      dtype='str')

In [8]:
# physical metrics for a single game — one row per (player × metric)
df_physical_metrics = client.query_game_physical_metrics(GAME_ID, possessions=["ALL"], as_dataframe=True).head()
df_physical_metrics.tail()

,location,season,possession,gameDate,playerPosition,player.id,player.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,team.id,team.name,metric.name,metric.p90,metric.raw
0,Home,2021-2022,ALL,2022-04-02,RB,1904,Joël Veltman,1,Premier League,14,Norwich City,4,Brighton & Hove Albion,accelerations,43.04,46.0
1,Home,2021-2022,ALL,2022-04-02,RB,1904,Joël Veltman,1,Premier League,14,Norwich City,4,Brighton & Hove Albion,athleticism_score,NaN,19.2
2,Home,2021-2022,ALL,2022-04-02,RB,1904,Joël Veltman,1,Premier League,14,Norwich City,4,Brighton & Hove Albion,decelerations,40.23,43.0
3,Home,2021-2022,ALL,2022-04-02,RB,1904,Joël Veltman,1,Premier League,14,Norwich City,4,Brighton & Hove Albion,game_appearances,NaN,1.0
4,Home,2021-2022,ALL,2022-04-02,RB,1904,Joël Veltman,1,Premier League,14,Norwich City,4,Brighton & Hove Albion,game_starts,NaN,1.0


In [9]:
# sprints — one row per sprint
df_sprints = client.get_game_sprints(GAME_ID, as_dataframe=True)
df_sprints.head()

,id,position,started,period,season,distance,videoUrl,gameDate,gameId,periodElapsedTimeEnd,...,yEnd,yStart,player.id,player.name,team.id,team.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name
0,4e9760f40219cadbf3c23929f6bf0716,CF,True,1,2021-2022,7.230348,https://epitome.gradientsports.com/film_room/b...,2022-04-02,1375,64.061261,...,25.400001,24.979199,458,Neal Maupay,4,Brighton & Hove Albion,1,Premier League,14,Norwich City
1,86e53cc3c6e98de65950449a17cb28b3,RCB,True,1,2021-2022,5.265535,https://epitome.gradientsports.com/film_room/b...,2022-04-02,1375,64.061261,...,18.650986,16.829787,545,Grant Hanley,14,Norwich City,1,Premier League,4,Brighton & Hove Albion
2,3e8f0c56f3932b4a84cd7c162c7ade46,DM,True,1,2021-2022,13.950935,https://epitome.gradientsports.com/film_room/b...,2022-04-02,1375,89.086286,...,27.100270,30.702025,6021,Mathias Normann,14,Norwich City,1,Premier League,4,Brighton & Hove Albion
3,9ebcfbf99c0b4b69276b5f64a04bd6d2,AM,True,1,2021-2022,13.005760,https://epitome.gradientsports.com/film_room/b...,2022-04-02,1375,97.094294,...,-17.661567,-4.821509,555,Kenny McLean,14,Norwich City,1,Premier League,4,Brighton & Hove Albion
4,27352b600e8105bac10f1b8209bec8e9,CF,True,1,2021-2022,6.768935,https://epitome.gradientsports.com/film_room/b...,2022-04-02,1375,104.101301,...,-21.020760,-26.364315,433,Danny Welbeck,4,Brighton & Hove Albion,1,Premier League,14,Norwich City


In [10]:
# high speed runs — one row per run
df_high_speed_runs = client.get_game_high_speed_runs(GAME_ID, as_dataframe=True)
df_high_speed_runs.head()

,id,position,started,period,season,distance,gameDate,gameId,shirtNumber,videoUrl,...,yEnd,yStart,player.id,player.name,competition.id,competition.name,opponentTeam.id,opponentTeam.name,team.id,team.name
0,ada47d78f232dd6f56b18727d89ed472,AM,True,1,2021-2022,5.452906,2022-04-02,1375,23,https://epitome.gradientsports.com/film_room/b...,...,31.964622,31.967204,555,Kenny McLean,1,Premier League,4,Brighton & Hove Albion,14,Norwich City
1,09b4151a6e360c49a837266252bb8e50,CF,True,1,2021-2022,4.788875,2022-04-02,1375,22,https://epitome.gradientsports.com/film_room/b...,...,21.381741,21.166246,565,Teemu Pukki,1,Premier League,4,Brighton & Hove Albion,14,Norwich City
2,4bf0198ae13dbc2434dd5dd8ba4c2c73,CF,True,1,2021-2022,5.163262,2022-04-02,1375,22,https://epitome.gradientsports.com/film_room/b...,...,20.353954,16.531526,565,Teemu Pukki,1,Premier League,4,Brighton & Hove Albion,14,Norwich City
3,dbc8b4590bf533d8c976c0196caa8e42,CF,True,1,2021-2022,5.019721,2022-04-02,1375,17,https://epitome.gradientsports.com/film_room/b...,...,-8.365465,-3.347063,5111,Milot Rashica,1,Premier League,4,Brighton & Hove Albion,14,Norwich City
4,32fe3fdf932c3475ae135e15b9f85dec,CF,True,1,2021-2022,18.692711,2022-04-02,1375,22,https://epitome.gradientsports.com/film_room/b...,...,-9.933483,-0.544806,565,Teemu Pukki,1,Premier League,4,Brighton & Hove Albion,14,Norwich City


In [13]:
# cross-player physical metrics — one row per (player × metric)
# Supported filter operators:
#   gt / lt          → {"operator": "gt",             "subject": "...", "value": <number>}
#   values_between   → {"operator": "values_between", "subject": "...", "values": [min, max]}
#   above_median     → NOT usable standalone; the API schema requires value/values for every operator
df_physical_metrics = client.query_physical_metrics(
        season="2024-2025",
        competition_ids=[1],
        possession="ALL",
        filters={
            "and": [
                {"operator": "gt", "subject": "game_appearances", "value": 10},
                {"operator": "gt", "subject": "sprints",          "value": 5},
            ]
        },
        as_dataframe=True,
    )
df_physical_metrics

,id,position,dob,firstName,lastName,playedHistory,team.id,team.name,metric.name,metric.p90,metric.p90Percentile,metric.raw,metric.rawPercentile
0,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,accelerations,40.2002,0.334572,4704.0000,0.918959
1,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,athleticism_score,NaN,NaN,42.5000,0.420074
2,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,decelerations,42.2512,0.278067,4944.0000,0.914498
3,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,game_appearances,NaN,NaN,110.0000,0.884758
4,1,CF,1993-07-28,Harry,Kane,"[{'competition_id': 1, 'competition_name': 'Pr...",160,FC Bayern München,game_starts,NaN,NaN,109.0000,0.931599
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23611,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,total_distance,9.2263,0.413383,22.0600,0.216357
23612,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,total_minutes,NaN,NaN,215.1900,0.208922
23613,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,visibility_percentage,NaN,NaN,0.3846,0.464684
23614,45549,LW,2008-08-29,Rio,Ngumoha,"[{'competition_id': 1, 'competition_name': 'Pr...",10,Liverpool,walk_distance,3.1075,0.540520,7.4300,0.213383


In [15]:
# player-level metrics — one row per (game × metric)
players = client.get_players(as_dataframe=True)
PLAYER_ID = players["id"].iloc[-1]

df_player_physical_metrics = client.query_player_physical_metrics(
        PLAYER_ID,
        seasons=["2024-2025"],
        competition_ids=[1],
        possessions=["ALL"],
        as_dataframe=True,
    ).head()

df_player_physical_metrics.tail()

""
